# Portfolio Farming: A Specialist-Module Approach to Kaggriculture

We expected four mathematically rigorous, domain-specific specialists to produce a competitively superior agent. Instead, combining these specialists created a new, catastrophic problem: coordination.

Throughout this project, we built monolithic agents optimizing for a single economic theory (Cournot, Linear Programming, Real Options). They repeatedly failed when environmental conditions shifted outside their primary domain. To solve this, we refactored the monolith into an isolated portfolio of specialists—an Agronomist, a Trader, an HR Manager, and a CFO—managed by a central Overseer. 

The immediate result was total systemic collapse.

This notebook documents exactly why. We demonstrate that **a component can be locally optimal and globally harmful.** A recommendation (like planting 100 crops) is not executable merely because it is locally profitable; it must also be globally feasible within payroll, cash, and physical constraints.

### What You Will Learn
1. **The Specialist vs. System Trap:** Why local optimization does not guarantee global feasibility.
2. **The Portfolio Architecture:** A reproducible method for isolating agent logic into functional domains.
3. **The Global Constraint Failure:** A step-by-step trace of how a minor seed purchase triggered total economic collapse.
4. **A Reusable Agent Framework:** A 10-step design pattern for building modular Kaggle agents that prioritize feasibility over raw optimization.


## 1. Evolution at a Glance

The following table summarizes our verifiable experimental journey. It clearly distinguishes local toy environments from actual leaderboard evaluations.


In [ ]:
import pandas as pd
from IPython.display import display, HTML

timeline = [
    {"Architecture": "Phase 3 (Monolith)", "Main Idea": "Simple melon snake farmer dumping inventory", "Evaluation": "Live Leaderboard", "Observed Result": "Score: 600", "Status": "Success (Baseline)", "Lesson": "Simplicity is robust but fragile to market crashes."},
    {"Architecture": "Phase 4b (Hungarian)", "Main Idea": "Task assignment via linear_sum_assignment", "Evaluation": "Live Leaderboard", "Observed Result": "Score: 330.9 - 348.0", "Status": "Failure", "Lesson": "Ignoring spatial density destroys execution efficiency."},
    {"Architecture": "Phase 6 (Opp. Cost)", "Main Idea": "Hungarian scheduler aware of opportunity cost", "Evaluation": "Local & Leaderboard", "Observed Result": "Local WR: 78.0% | LB: 504.7", "Status": "Mixed", "Lesson": "Local dominance does not translate if market-crash dynamics are ignored."},
    {"Architecture": "Phase 9 (Real Options)", "Main Idea": "Newsvendor + Real Options parameter sweep", "Evaluation": "Local Test Pool", "Observed Result": "Win Rate: 0.0%", "Status": "Failure", "Lesson": "Adversarial supply registers as negative demand, halting all production."},
    {"Architecture": "Phase 12 (Portfolio)", "Main Idea": "4 independent specialists + 1 Overseer", "Evaluation": "Local Gauntlet (320 Eps)", "Observed Result": "Win Rate: 0.0%", "Status": "Failure", "Lesson": "A component can be locally optimal but globally harmful without feasibility constraints."},
    {"Architecture": "Phase 12b (Format Fix)", "Main Idea": "Corrected BUY_SEED / BUY_ANIMAL command format", "Evaluation": "Local Gauntlet (320 Eps)", "Observed Result": "Scores: -77.9, -49.2, 28.0", "Status": "Mixed", "Lesson": "Malformed commands silently aborting a full turn's market actions mimic strategic failure."}
]

df = pd.DataFrame(timeline)
display(HTML(df.to_html(index=False)))


## 2. The Illusion of the Specialist

Consider the isolated recommendations of our four specialists:

*   **Crop Specialist:** "Wheat margins are high. Plant 160 wheat."
*   **Labor Specialist:** "Workload is massive. We need 16 workers."
*   **Capital Specialist:** "Land is cheap relative to yield. Buy a new plot."
*   **Market Specialist:** "Prices are dropping. Halt sales to throttle supply."

Individually, every single one of these recommendations is mathematically reasonable and locally optimal. 

Combined, they guarantee failure. The farm attempts to buy land, buy seeds, and hire labor simultaneously while refusing to sell inventory. The result is instant bankruptcy.

This is fundamentally a systems and resource-allocation problem. We introduce the concept of **Global Feasibility**. A recommendation is not executable merely because it is locally profitable. It must satisfy strict, overlapping constraints:
*   Cash constraints
*   Payroll requirements
*   Worker physical capacity
*   Time constraints
*   Inventory/Preconditions

This is the deeper lesson of the portfolio experiment: optimization without global feasibility is just a complicated way to crash.


## 3. The Architecture Visualized

To build this agent, we decomposed the farm into four quadrants, feeding into an Overseer.

*Note: The visual below is drawn natively using Matplotlib to ensure clean rendering inside the Kaggle environment.*


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')

# Colors
c_bg = '#f8f9fa'
c_edge = '#495057'
c_specialist = '#e9ecef'
c_overseer = '#dee2e6'
c_engine = '#e3f2fd'
c_text = '#212529'

def draw_box(x, y, w, h, text, color, edge=c_edge):
    rect = patches.Rectangle((x, y), w, h, facecolor=color, edgecolor=edge, linewidth=2, zorder=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=11, fontweight='bold', color=c_text, zorder=3)

def draw_arrow(x1, y1, x2, y2, text=None):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(facecolor=c_edge, width=2, headwidth=10, shrink=0.05), zorder=1)
    if text:
        mid_x, mid_y = (x1+x2)/2, (y1+y2)/2
        ax.text(mid_x + 0.2, mid_y, text, ha='left', va='center', fontsize=9, bbox=dict(facecolor='white', edgecolor='none', alpha=0.8))

# Draw Diagram 1: Portfolio Architecture
draw_box(3, 9, 4, 1, 'Game State (Observation)', c_bg)

draw_box(1, 7, 2, 1, 'Crop Specialist\n(Agronomist)', c_specialist)
draw_box(3.5, 7, 2, 1, 'Market Specialist\n(Trader)', c_specialist)
draw_box(6, 7, 2, 1, 'Labor Specialist\n(HR)', c_specialist)
draw_box(8.5, 7, 2, 1, 'Capital Specialist\n(CFO)', c_specialist)

draw_arrow(5, 9, 2, 8)
draw_arrow(5, 9, 4.5, 8)
draw_arrow(5, 9, 7, 8)
draw_arrow(5, 9, 9.5, 8)

draw_box(3, 5, 4, 1, 'Portfolio Overseer\n(Conflict Resolution)', c_overseer)

draw_arrow(2, 7, 4, 6, "Target Crops")
draw_arrow(4.5, 7, 4.5, 6, "Sell Orders")
draw_arrow(7, 7, 5.5, 6, "Payroll Req")
draw_arrow(9.5, 7, 6, 6, "Cap Req")

draw_box(3, 3, 4, 1, 'Execution Engine', c_engine)
draw_arrow(5, 5, 5, 4, "Feasible Global Plan")

draw_box(3, 1, 4, 1, 'New Game State', c_bg)
draw_arrow(5, 3, 5, 2, "Action Submitted")

plt.title("Diagram 1: Portfolio Architecture", fontsize=14, fontweight='bold', pad=20)
plt.xlim(0, 11.5)
plt.ylim(0, 10.5)
plt.show()


## 3b. Specialist Decision Flow

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 6))
ax.axis('off')

# Colors
c_bg = '#f8f9fa'
c_edge = '#495057'
c_model = '#e9ecef'
c_check = '#dee2e6'
c_action = '#e3f2fd'
c_text = '#212529'

def draw_box(x, y, w, h, text, color):
    rect = patches.Rectangle((x, y), w, h, facecolor=color, edgecolor=c_edge, linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=11, fontweight='bold', color=c_text)

def draw_arrow(x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(facecolor=c_edge, width=2, headwidth=10, shrink=0.05))

draw_box(2.5, 9, 3, 1, 'Observation', c_bg)
draw_arrow(4, 9, 4, 7.5)
draw_box(2.5, 6.5, 3, 1, 'Specialist Model', c_model)
draw_arrow(4, 6.5, 4, 5)
draw_box(2.5, 4, 3, 1, 'Recommendation', c_bg)
draw_arrow(4, 4, 4, 2.5)
draw_box(1.5, 1.5, 5, 1, 'Constraint Check', c_check)
draw_arrow(4, 1.5, 4, 0)
draw_box(2.5, -1, 3, 1, 'Accept/Modify/Reject', c_bg)
draw_arrow(4, -1, 4, -2.5)
draw_box(2.5, -3.5, 3, 1, 'Execution', c_action)

plt.title("Diagram 2: Specialist Decision Flow", fontsize=14, fontweight='bold', pad=20)
plt.xlim(0, 8)
plt.ylim(-4.5, 10.5)
plt.show()


## 3c. The Failure Cascade

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 8))
ax.axis('off')

# Colors
c_edge = '#495057'
c_text = '#212529'
c_node = '#f8f9fa'

steps = [
    "Crop Target",
    "Large Seed Purchase",
    "Cash Drain",
    "Insufficient Payroll",
    "Insufficient Labor",
    "Unmanaged Tasks",
    "Care Lapses",
    "Economic Collapse"
]

y = 8
for i, step in enumerate(steps):
    rect = patches.Rectangle((2.5, y-0.5), 3, 1, facecolor=c_node, edgecolor=c_edge, linewidth=2)
    ax.add_patch(rect)
    ax.text(4, y, step, ha='center', va='center', fontsize=10, fontweight='bold', color=c_text)
    
    if i < len(steps) - 1:
        ax.annotate('', xy=(4, y-0.5), xytext=(4, y-1.5),
                    arrowprops=dict(facecolor=c_edge, width=2, headwidth=10, shrink=0.05))
    y -= 1.5

plt.title("Diagram 3: The Failure Cascade", fontsize=14, fontweight='bold', pad=20)
plt.xlim(0, 8)
plt.ylim(-3.5, 9)
plt.show()


## 4. The Specialists in Isolation

To understand the system failure, we must first understand the components. 

Below is the evaluation of each specialist in strict isolation. 

> **How to Read These Numbers:** The "Isolated Score" below is a **single-episode final-money result** evaluated under a fixed protocol (1 episode, 360 steps, vs. a dummy `PASS` opponent, Seed 42). These are *not* multi-episode win rates, and they are *not* comparable to Kaggle Leaderboard scores. The naive baseline under these exact conditions achieved a final money score of **3199.0**.

### 1. The Agronomist (Crop Specialist)
*   **What problem does it solve?** Deciding which seeds to plant to maximize future profit without crashing the market.
*   **What information does it use?** Current market prices, opponent crop configurations, and crop maturation times.
*   **What does it output?** Target tile allocations per crop.
*   **What technique does it use?** Cournot Best-Response. It forecasts the opponent's supply 3 days ahead, predicts the market price drop using the known demand curve, and sizes our production to the exact point where marginal revenue equals marginal cost.
*   **What did the experiment measure?** Final money generated when only the Agronomist directed planting (Baseline handled other tasks).
*   **What did we learn?** `Isolated Score: 226.0`. Relying purely on deep forecasting without understanding spatial efficiency led to massive underperformance.
*   **Where can it fail?** It generates targets, but blindly ignores whether the farm actually has the labor capacity or cash to execute those targets.

### 2. The Trader (Market Specialist)
*   **What problem does it solve?** Preventing self-inflicted price crashes caused by dumping inventory.
*   **What information does it use?** Shed inventory, current market parameters, and marginal seed costs.
*   **What does it output?** Exact sell quantities for the current hour.
*   **What technique does it use?** Marginal Revenue `R(q)` Throttling. It iteratively simulates the market price if it were to sell `q+1` units, halting the sale exactly when the simulated price drops below the marginal cost of production.
*   **What did the experiment measure?** Final money when only the Trader managed sales.
*   **What did we learn?** `Isolated Score: 3170.0`. It roughly matched the baseline. Market throttling is defensive, not offensive.
*   **Where can it fail?** By refusing to sell at a loss, it can hoard inventory, starving the farm of operating cash needed for today's payroll.

### 3. The HR Manager (Labor Specialist)
*   **What problem does it solve?** Assigning workers safely to spatial tasks and managing headcount.
*   **What information does it use?** Current worker positions, empty tiles, plant states, and pending volume.
*   **What does it output?** Unit `(x,y)` target vectors and `HIRE` orders.
*   **What technique does it use?** Precondition-safe Task Assignment. It frames the farm as a bipartite graph and uses `scipy.optimize.linear_sum_assignment` to minimize total travel distance, while separately scaling hiring budget to the pending volume queue.
*   **What did the experiment measure?** Final money when only the HR Manager handled execution.
*   **What did we learn?** `Isolated Score: 4611.0`. The only module to significantly outperform the baseline. Mathematical routing efficiency is the strongest local driver of profit.
*   **Where can it fail?** It assumes any `HIRE` order it generates will successfully clear the market, oblivious to whether the CFO or Agronomist spent the money first.

### 4. The CFO (Capital Specialist)
*   **What problem does it solve?** Timing the purchase of permanent assets (Land and Livestock).
*   **What information does it use?** Available cash, current day, current land size.
*   **What does it output?** Capital `BUY` orders.
*   **What technique does it use?** Shadow-Price Linear Programming. It runs a continuous relaxation LP of the farm to extract the dual shadow prices (the marginal value of an extra $1 vs an extra tile of land).
*   **What did the experiment measure?** Final money when only the CFO bought assets.
*   **What did we learn?** `Isolated Score: 2201.0`. Premature capital lockup severely hurts compounding growth.
*   **Where can it fail?** It locks up cash in illiquid assets without checking if the farm needs that cash to buy seeds tomorrow.


## 5. Why the Specialists Failed to Compose

The portfolio architecture failed because it confused **Local Optimization** with **Global Optimization**.

The Crop Specialist optimized crop economics. The Labor Specialist optimized task execution. The Capital Specialist optimized capital allocation. But none of these objectives automatically guarantees system-level feasibility.

> **The Core Rule:** Local Objective + Local Constraints $\neq$ Globally Feasible Plan.

### A Toy Example (For Illustration Only)
*Note: This is a simplified numerical example, not a direct Kaggriculture log.*

Assume the farm has `$100`.
1.  **CFO Analysis:** Land costs `$80`. It yields a 20% ROI. Local constraint passed (`$80 < $100`). **Action:** Buy Land.
2.  **Agronomist Analysis:** We need `$30` of seeds to plant the new land. Local constraint passed (`$30 < $100`). **Action:** Buy Seeds.

**The Global Result:** The agent executes both. Cost = `$110`. The engine throws an `Insufficient Funds` error, the turn is wasted, and the farm falls behind. Both specialists were locally correct; the system was globally unfeasible.

### Failure Mode Catalog

Below is a catalog of real, documented failures we encountered while developing this architecture. If you are debugging your own agent, check for these:

| Failure Mode | Symptom | Root Cause | Detection Signal | Fix |
| :--- | :--- | :--- | :--- | :--- |
| **Precondition Violation** | `FEED` action fails repeatedly. | Solver commanded `FEED` without verifying wheat inventory. | High error counts in Kaggle replays; animals starving despite idle hands. | Wrap all actions in hard game-state preconditions before passing to the solver. |
| **Market Flooding** | Massive crop yields result in minimal revenue. | Agent dumped entire premium inventory (e.g. Melons) in one tick. | Realized sell price was vastly lower than expected marginal revenue. | Implement throttled selling based on real-time `R(q)` estimates. |
| **Distance Penalty Interference** | Workers walk past dying crops to harvest distant ones. | Cost function prioritized high-value distant harvests over urgent adjacent care. | High "care lapses" in local gauntlet metrics. | Implement a strict execution hierarchy (`FEED > CARE > HARVEST`), checking distance secondarily. |
| **Capital Drain** | Agent goes bankrupt on Day 5. | Static script hired too many hands too early before revenue materialized. | Final money < 0; agent stops taking actions. | Workload-driven hiring; cap max daily hires against available capital. |
| **Command Format Silently Dropped** | Agent submits market orders, but inventory never updates; full turn's market actions are aborted. | Submitting `["BUY_SEED", "WHEAT"]` instead of `["BUY_SEED", "WHEAT", 1]`. The engine silently drops malformed market orders. | Submitted orders do not reflect in the next turn's inventory or cash balance. | Ensure all market commands strictly match the engine specification (`BUY_SEED <crop> <n>`, `BUY_ANIMAL <animal> <n>`). |


## 6. Real Conflict Resolution Traces

To see this play out in reality, we traced the Overseer's decision logic during actual execution. 

### Trace 1: The Seed-Budget Failure (The Fatal Flaw)
This is the specific interaction that crashed our gauntlet run to a 0.0% win rate against the Melon Dumper. The Overseer successfully protected capital, but failed to impose a global feasibility constraint on production inputs.

```text
======================================================
[SCENARIO: Adversarial Market Response vs Melon Dumper]
[DAY 0, HOUR 0]
======================================================
Observed State: Opponent planting heavy melons.
↓
Crop Specialist: "Melon margin is 0. Pivot to Wheat. Target: 167 Wheat."
↓
Overseer Action: Buy 160 Wheat Seeds. (Cost: $3200)
↓
Capital Remaining: $0  (Starting cash drained).
↓
Labor Specialist: "Massive planting volume queued. Hire 16 hands."
↓
Payroll Requirement: $160
↓
Overseer Action: Submit HIRE orders.
↓
Game Engine Response: REJECTED (Insufficient Funds).
↓
Actual Available Labor: 1 Farmer.
↓
Final Outcome: The lone farmer cannot plant or water 160 tiles. 
Resulting in 471 Care Lapses and Total Economic Collapse.
```
**Lesson:** The Overseer protected capital purchases but did not impose the same global feasibility constraint on production inputs.

### Trace 2: Successful Capital Deferment
While the seed logic failed, the capital logic actually worked as designed. (Note: Only one sufficiently documented trace was available from our logs for a successful deferment).

```text
======================================================
[SCENARIO: Early Game Expansion vs Naive Cluster]
[DAY 2, HOUR 0]
======================================================
Observed State: $1450 Cash. Pending harvest queue.
↓
Capital Specialist: "ROI positive on Land and Cow. Buy both." (Cost: $1400)
↓
Labor Specialist: "Harvest queue requires 6 hands to clear."
↓
Payroll Requirement: $60
↓
Conflict: Cash ($1450) - Capital ($1400) = $50. Payroll ($60) is unfeasible.
↓
Overseer Decision: REJECT Capital Specialist. Defer by one cycle.
↓
Observed Outcome: Payroll clears. Hands hired. Harvest saved.
```
**Lesson:** Explicit priority rules work, but they must cover *all* expenditure avenues.


## 7. The Gauntlet Evaluation

We evaluated the complete portfolio architecture across 320 episodes (80 episodes per archetype). 

*Evaluation Protocol:* 80 episodes against fixed archetype opponents, standard game length.

| Opponent (Archetype) | Episodes | Wins | Losses | Ties | Win Rate | Care Lapses/Game | Avg Hands/Day |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Melon Dumper** | 80 | 0 | 80 | 0 | **0.0%** | 471 | 0.01 |
| **Diversified** | 80 | 0 | 80 | 0 | **0.0%** | 426 | 0.01 |
| **Land Rush** | 80 | 0 | 80 | 0 | **0.0%** | 420 | 0.01 |
| **Hoarder** | 80 | 0 | 80 | 0 | **0.0%** | 490 | 0.01 |

**Interpretation:** The portfolio architecture did not improve competitive performance in this evaluation. It crashed the agent. 

**Why?** As detailed in Trace 1, the failure to subject the Crop Specialist's massive seed requests to a global budget check caused the agent to instantly bankrupt itself on Day 0, resulting in average hands/day of 0.01 (meaning it almost never successfully hired a hand) and ~450+ unwatered plants per game.


## 8. What We Got Wrong

This failure was an engineering mistake, not a failure of the algorithms.

1.  **We confused local specialist quality with system quality.** A perfect shadow-price LP and a perfect Hungarian router are useless if they cannot coordinate their checking accounts.
2.  **We optimized specialists independently.** We tested their math in isolation, blind to how their edge cases would interact.
3.  **We protected some resources, but not all resources.** The Overseer checked if land purchases starved the payroll, but completely forgot to check if *seed* purchases starved the payroll.
4.  **We lacked a global feasibility layer.** The Overseer acted as a pass-through router rather than a strict budget enforcer.


## 9. A Practical Specialist-Agent Design Pattern

To prevent others from making this exact mistake, here is a 10-step reusable framework for designing modular Kaggriculture agents.

**Step 1 — Identify independent decision domains.** (e.g., Crops, Market, Labor, Capital).
**Step 2 — Define each specialist's objective.** (e.g., Labor minimizes travel distance).
**Step 3 — Define specialist inputs and outputs.** (e.g., Inputs: State, Outputs: Requested Orders).
**Step 4 — Prevent specialists from directly executing actions.** No module should return actions to the game engine directly.
**Step 5 — Add a central coordinator (The Overseer).** 
**Step 6 — Define GLOBAL constraints.** (Total Cash, Total Execution Time, Total Worker Capacity, Total Inventory).
**Step 7 — Add feasibility checks.** The Overseer must sum the resource requirements of *all* specialists before execution.
**Step 8 — Resolve conflicts.** Establish explicit priority (e.g., Payroll > Seeds > Land).
**Step 9 — Execute only feasible actions.** Drop the lowest priority orders until the plan fits the budget.
**Step 10 — Evaluate both components AND the complete system.** 

### The Debugging Checklist
Before submitting a modular agent, ask these questions:

**ECONOMY**
*   [ ] Can we afford the sum total of all planned purchases?
*   [ ] Is payroll explicitly protected from aggressive capital/seed buys?
*   [ ] Is operating cash protected from hoarding inventory?

**PRODUCTION & WORKERS**
*   [ ] Are production targets bounded by our physical execution capacity?
*   [ ] Can we actually water/feed everything we plant/buy today?
*   [ ] Are workers available, or did our hires get rejected?

**SYSTEM & EVALUATION**
*   [ ] Can one specialist's recommendation make another specialist's action impossible?
*   [ ] Are global constraints checked *before* returning actions to the engine?
*   [ ] Are local tests (single episode) and leaderboard results clearly separated?
*   [ ] Are all market commands strictly matching the engine specification (e.g., `["BUY_SEED", "WHEAT", 1]`), avoiding malformed commands that silently abort a full turn's market actions?


## 10. What We Would Test Next

Based directly on the observed failure, our future experiments will focus entirely on coordination. 

*   **Global Cash-Flow Constraint:** 
    *   *Hypothesis:* Summing all requested expenses (Payroll + Seeds + Capital) and aggressively filtering them against available cash will prevent Day 0 bankruptcy.
    *   *Implementation:* Centralized `budget_manager` in the Overseer.
    *   *Measurement:* Average hands/day rising from 0.01 to >2.
*   **Workload-Aware Production Cap:**
    *   *Hypothesis:* Capping crop targets by `current_headcount * 8` will prevent the Agronomist from creating physically unmanageable farms.
    *   *Implementation:* Pass `max_tiles` limit to the Crop Specialist.
    *   *Failure Criterion:* Win rate does not improve despite reduced care lapses.

## 11. Conclusion

**Specialization solved one problem and created another: coordination.**

We proved that decomposing the Kaggriculture problem into distinct mathematical domains (Cournot, Hungarian, LP) is feasible and results in highly competent local decision-making. However, the experiment established that local competency is dangerous without a strict global feasibility constraint.

The next version of this architecture will not focus on building better specialists. It will make the coordinator completely responsible for global feasibility.


## 12. Reproducibility Guide

You can reproduce the exact failures and isolation tests locally. Ensure the 4 specialist scripts and `portfolio_overseer.py` are in your directory. *(Note: These scripts are part of our internal repo and may require adaptation to run outside of our exact folder structure).*

**1. Environment Setup**
```bash
pip install kaggle-environments scipy numpy
```

**2. Specialist Isolation Evaluation**
Run this to verify the single-episode scores (generates the 226/3170/4611/2201 scores against the naive baseline).
```bash
python test_specialists.py
```

**3. Failure Reproduction (The Seed-Budget Crash)**
Run this to trace the exact Day 0 cash drain described in Trace 1.
```bash
python test_overseer_single.py
```
